In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Install Dependencies

In [2]:
!pip install num2words bert_score transformers -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.4 MB/s eta 0:00:00


## 2. Imports

In [3]:
import os
import json
import ast
import random

import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader

from transformers import BertTokenizerFast, BertModel
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from bert_score import score as bert_score_fn
import nltk

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

from warnings import filterwarnings
filterwarnings('ignore')

In [4]:
# ── Paths ────────────────────────────────────────────────────────────────────
BASE          = "/content/drive/MyDrive/ITAITA_Project"

CONFIG_PATH   = os.path.join(BASE, "Config")
MODEL_RESULTS = os.path.join(BASE, "Model_Results")   # read weights/configs from here
DATA_DIR      = os.path.join(BASE, "Data")

MASTER_CSV_PATH = os.path.join(DATA_DIR, "master_encoded_data.csv")
TEST_DATA_PATH  = os.path.join(CONFIG_PATH, "test_data_with_color_info.csv")

OLD_PREFIX = "/content/drive/Othercomputers/My Laptop/Desktop/ITAITA_Project/Data"
NEW_PREFIX = DATA_DIR

# ── Output folder (writable) ─────────────────────────────────────────────────
OUTPUT_DIR = "/content/drive/MyDrive/ITAITA_Eval_Results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RUN_NAMES = [
    'ShallowCNN_simple', 'ShallowCNN_bert',
    'DeepCNN_simple',    'DeepCNN_bert',
    'ResNetCNN_simple',  'ResNetCNN_bert',
]

DATASETS = ['Shapes_Data', 'Numbers_Data', 'TicTacToe_Data']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [5]:
# ── CNN Architectures ────────────────────────────────────────────────────────

class ShallowCNN(nn.Module):
    def __init__(self, embed_dim=256):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, 3, padding=1)
        self.conv2 = nn.Conv2d(64, 256, 3, padding=1)
        self.proj  = nn.Conv2d(256, embed_dim, 1)

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        x = self.proj(x)
        return x.flatten(2).permute(0, 2, 1)


class DeepCNN(nn.Module):
    def __init__(self, embed_dim=256):
        super().__init__()
        self.conv1 = nn.Conv2d(3,   32,  3, padding=1)
        self.conv2 = nn.Conv2d(32,  64,  3, padding=1)
        self.conv3 = nn.Conv2d(64,  128, 3, padding=1)
        self.conv4 = nn.Conv2d(128, 256, 3, padding=1)
        self.conv5 = nn.Conv2d(256, 256, 3, padding=1)
        self.proj  = nn.Conv2d(256, embed_dim, 1)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.conv5(x))
        x = self.proj(x)
        return x.flatten(2).permute(0, 2, 1)


class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn1   = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn2   = nn.BatchNorm2d(channels)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + x)


class ResNetCNN(nn.Module):
    def __init__(self, embed_dim=256):
        super().__init__()
        self.conv1 = nn.Conv2d(3,   64,  3, padding=1)
        self.res1  = ResidualBlock(64)
        self.conv2 = nn.Conv2d(64,  256, 3, padding=1)
        self.res2  = ResidualBlock(256)
        self.proj  = nn.Conv2d(256, embed_dim, 1)

    def forward(self, x):
        x = self.res1(self.conv1(x))
        x = F.max_pool2d(x, 2)
        x = self.res2(self.conv2(x))
        x = F.max_pool2d(x, 2)
        x = self.proj(x)
        return x.flatten(2).permute(0, 2, 1)


CNN_CLASSES = {
    'ShallowCNN': ShallowCNN,
    'DeepCNN':    DeepCNN,
    'ResNetCNN':  ResNetCNN,
}

In [6]:
# ── Transformer Decoder ──────────────────────────────────────────────────────

class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, max_length=150, num_heads=8,
                 N_layers=3, ff_dim=512, dropout=0.1, use_bert=False):
        super().__init__()
        self.use_bert = use_bert

        if use_bert:
            bert = BertModel.from_pretrained("bert-base-uncased")
            self.bert_embeddings = bert.embeddings
            self.bert_proj = nn.Linear(768, embed_dim)
        else:
            self.embedding     = nn.Embedding(vocab_size, embed_dim)
            self.pos_embedding = nn.Embedding(max_length, embed_dim)

        self.embed_dim  = embed_dim
        decoder_layer   = nn.TransformerDecoderLayer(embed_dim, num_heads, ff_dim, dropout, batch_first=True)
        self.transformer = nn.TransformerDecoder(decoder_layer, num_layers=N_layers)
        self.fc_out     = nn.Linear(embed_dim, vocab_size)

    def forward(self, tgt, memory, attention_mask=None):
        seq_len = tgt.shape[1]
        if self.use_bert:
            x = self.bert_proj(self.bert_embeddings(input_ids=tgt))
            tgt_key_padding_mask = (attention_mask == 0) if attention_mask is not None else None
        else:
            positions = torch.arange(seq_len, device=tgt.device)
            x = self.embedding(tgt) + self.pos_embedding(positions)
            tgt_key_padding_mask = None

        mask = nn.Transformer.generate_square_subsequent_mask(seq_len, device=tgt.device)
        x    = self.transformer(x, memory, tgt_mask=mask, tgt_key_padding_mask=tgt_key_padding_mask)
        return self.fc_out(x)

## 5. Load Tokeniser & Vocabulary

In [7]:
bert_tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

with open(os.path.join(CONFIG_PATH, "word2idx.json")) as f:
    word2idx = json.load(f)
with open(os.path.join(CONFIG_PATH, "idx2word.json")) as f:
    idx2word = {int(k): v for k, v in json.load(f).items()}

print(f"Vocabulary size: {len(word2idx)}")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Vocabulary size: 172


In [8]:
# Load from cached test CSV if available, otherwise fall back to master CSV
try:
    df = pd.read_csv(TEST_DATA_PATH)
    print(f"Loaded test CSV from {TEST_DATA_PATH}")
except FileNotFoundError:
    df = pd.read_csv(MASTER_CSV_PATH)
    print(f"Loaded master CSV from {MASTER_CSV_PATH}")

test_data = df[df['split'] == 'test'].reset_index(drop=True)

# Remap file paths from original local machine paths to current environment
test_data['file_path'] = test_data['file_path'].str.replace(OLD_PREFIX, NEW_PREFIX, regex=False)

print(f"Test samples: {len(test_data)}")
print(test_data['dataset'].value_counts())

Loaded test CSV from /content/drive/MyDrive/ITAITA_Project/Config/test_data_with_color_info.csv
Test samples: 2238
dataset
TicTacToe_Data    750
Numbers_Data      744
Shapes_Data       744
Name: count, dtype: int64


In [9]:
# Add colour/BW flag if not already present
def detect_colour(path):
    try:
        return Image.open(path).mode == 'RGB'
    except Exception:
        return None

if 'is_colour' not in test_data.columns:
    print("Detecting colour/BW (this may take a moment)...")
    test_data['is_colour'] = test_data['file_path'].apply(detect_colour)

print(test_data['is_colour'].value_counts())

is_colour
False    1124
True     1114
Name: count, dtype: int64


## 7. Image Transform & Generate Function

In [10]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.Lambda(lambda img: img.convert('RGB')),
    transforms.ToTensor(),
])

def load_image(path):
    return transform(Image.open(path))


def generate(cnn, decoder, image, device, max_length=150,
             word2idx=None, idx2word=None, tokenizer=None):
    cnn.eval()
    decoder.eval()
    use_bert = tokenizer is not None

    start_token = tokenizer.cls_token_id if use_bert else word2idx['<START>']
    end_token   = tokenizer.sep_token_id if use_bert else word2idx['<END>']

    with torch.no_grad():
        memory = cnn(image.unsqueeze(0).to(device))
        tokens = [start_token]

        for _ in range(max_length):
            tokens_tensor  = torch.tensor(tokens).unsqueeze(0).to(device)
            attention_mask = torch.ones_like(tokens_tensor) if use_bert else None
            output         = decoder(tokens_tensor, memory, attention_mask=attention_mask)
            next_token     = torch.argmax(output[0, -1, :]).item()
            if next_token == end_token:
                break
            tokens.append(next_token)

    if use_bert:
        return tokenizer.decode(tokens[1:], skip_special_tokens=True)
    else:
        return ' '.join(idx2word[t] for t in tokens[1:])

In [11]:
for run_name in RUN_NAMES:
    save_dir  = os.path.join(MODEL_RESULTS, run_name)
    pred_path = os.path.join(OUTPUT_DIR, f"{run_name}_predictions.csv")  # ← save here
    print(f"\n{'─'*60}")
    print(f"Generating: {run_name}")

    with open(os.path.join(save_dir, "config.json")) as f:
        config = json.load(f)

    encoding   = config['encoding']
    cnn_arch   = config['cnn_architecture']
    vocab_size = config['vocab_size']

    cnn_model = CNN_CLASSES[cnn_arch]().to(device)
    dec_model = (
        TransformerDecoder(vocab_size=bert_tokenizer.vocab_size, use_bert=True).to(device)
        if encoding == 'bert'
        else TransformerDecoder(vocab_size=vocab_size).to(device)
    )

    checkpoint = torch.load(os.path.join(save_dir, "best_model.pt"), map_location=device)
    cnn_model.load_state_dict(checkpoint['cnn'])
    dec_model.load_state_dict(checkpoint['decoder'])
    cnn_model.eval()
    dec_model.eval()
    print(f"  Weights loaded ✅  encoding={encoding}, arch={cnn_arch}")

    records = []
    for _, row in tqdm(test_data.iterrows(), total=len(test_data), desc=run_name):
        image = load_image(row['file_path'])
        pred  = (
            generate(cnn_model, dec_model, image, device, tokenizer=bert_tokenizer)
            if encoding == 'bert'
            else generate(cnn_model, dec_model, image, device, word2idx=word2idx, idx2word=idx2word)
        )
        records.append({
            'file_path':  row['file_path'],
            'dataset':    row['dataset'],
            'is_colour':  row['is_colour'],
            'reference':  row['text'],
            'prediction': pred,
        })

    pd.DataFrame(records).to_csv(pred_path, index=False)
    print(f"  Saved → {pred_path}")

print(f"\n{'='*60}")
print("All predictions generated ✅")


────────────────────────────────────────────────────────────
Generating: ShallowCNN_simple
  Weights loaded ✅  encoding=simple, arch=ShallowCNN


ShallowCNN_simple:   6%|▋         | 144/2238 [01:59<28:54,  1.21it/s]


KeyboardInterrupt: 

## 9. Metric Functions

In [12]:
smoothie = SmoothingFunction().method1

def exact_match(ref, hyp):
    return float(ref.strip() == hyp.strip())

def bleu4(ref, hyp):
    ref_tokens = ref.strip().split()
    hyp_tokens = hyp.strip().split()
    if not hyp_tokens:
        return 0.0
    return sentence_bleu([ref_tokens], hyp_tokens,
                         weights=(0.25, 0.25, 0.25, 0.25),
                         smoothing_function=smoothie)

def meteor(ref, hyp):
    ref_tokens = ref.strip().split()
    hyp_tokens = hyp.strip().split()
    if not hyp_tokens:
        return 0.0
    return meteor_score([ref_tokens], hyp_tokens)

def compute_bertscore_batch(refs, hyps):
    _, _, F1 = bert_score_fn(hyps, refs, lang='en', verbose=False)
    return F1.tolist()

def score_predictions(df):
    refs = df['reference'].fillna('').tolist()
    hyps = df['prediction'].fillna('').tolist()
    df   = df.copy()
    print("  Computing Exact Match, BLEU-4, METEOR...")
    df['exact_match'] = [exact_match(r, h) for r, h in zip(refs, hyps)]
    df['bleu4']       = [bleu4(r, h)       for r, h in zip(refs, hyps)]
    df['meteor']      = [meteor(r, h)       for r, h in zip(refs, hyps)]
    print("  Computing BERTScore...")
    df['bertscore']   = compute_bertscore_batch(refs, hyps)
    return df

def aggregate(df):
    return {
        'exact_match': round(df['exact_match'].mean(), 4),
        'bleu4':       round(df['bleu4'].mean(), 4),
        'meteor':      round(df['meteor'].mean(), 4),
        'bertscore':   round(df['bertscore'].mean(), 4),
        'n':           len(df),
    }

## 10. Score All Models
Computes metrics overall and broken down by dataset and colour/BW.

In [15]:
all_results = {}

for run_name in RUN_NAMES:
    pred_path = os.path.join(OUTPUT_DIR, f"{run_name}_predictions.csv")
    if not os.path.exists(pred_path):
        print(f"[MISSING] {run_name} — run Cell 8 first")
        continue

    print(f"\n{'─'*60}")
    print(f"Scoring: {run_name}")
    df      = pd.read_csv(pred_path)
    df      = score_predictions(df)
    results = {}

    results['overall'] = aggregate(df)

    for ds in DATASETS:
        subset = df[df['dataset'] == ds]
        results[ds]              = aggregate(subset)
        results[f'{ds}_colour']  = aggregate(subset[subset['is_colour'] == True])
        results[f'{ds}_bw']      = aggregate(subset[subset['is_colour'] == False])

    all_results[run_name] = results
    print(f"  Done ✅  BLEU-4={results['overall']['bleu4']}  "
          f"METEOR={results['overall']['meteor']}  BERTScore={results['overall']['bertscore']}")

# Save full results JSON
out_path = "/content/drive/MyDrive/ITAITA_Eval_Results/evaluation_results.json"
with open(out_path, 'w') as f:
    json.dump(all_results, f, indent=2)
print(f"\nResults saved → {out_path}")


────────────────────────────────────────────────────────────
Scoring: ShallowCNN_simple
  Computing Exact Match, BLEU-4, METEOR...
  Computing BERTScore...


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Done ✅  BLEU-4=0.1895  METEOR=0.4677  BERTScore=0.9172

────────────────────────────────────────────────────────────
Scoring: ResNetCNN_bert
  Computing Exact Match, BLEU-4, METEOR...
  Computing BERTScore...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Done ✅  BLEU-4=0.2371  METEOR=0.504  BERTScore=0.9299

Results saved → /content/drive/MyDrive/ITAITA_Eval_Results/evaluation_results.json


## 11. Results Table

In [16]:
from IPython.display import display

rows = []
for run in RUN_NAMES:
    if run not in all_results:
        continue
    d = all_results[run]['overall']
    rows.append({
        'Model':       run.replace('_', ' — ', 1),
        'Exact Match': d['exact_match'],
        'BLEU-4':      d['bleu4'],
        'METEOR':      d['meteor'],
        'BERTScore':   d['bertscore'],
    })

results_df = pd.DataFrame(rows).set_index('Model')

display(
    results_df.style
      .format('{:.4f}')
      .highlight_max(axis=0, color='#d4edda')
      .highlight_min(axis=0, color='#fde8e8')
      .set_caption('Overall results — all test samples')
      .set_table_styles([{
          'selector': 'caption',
          'props': [('font-size', '14px'), ('font-weight', 'bold'), ('padding-bottom', '8px')]
      }])
)

,Exact Match,BLEU-4,METEOR,BERTScore
Model,,,,
ShallowCNN — simple,0.0000,0.1398,0.4064,0.9086
ShallowCNN — bert,0.0000,0.1542,0.4236,0.9216
DeepCNN — simple,0.0000,0.1987,0.4807,0.9183
DeepCNN — bert,0.0004,0.2190,0.4994,0.9292
ResNetCNN — simple,0.0000,0.1895,0.4677,0.9172
ResNetCNN — bert,0.0036,0.2371,0.5040,0.9299
